# Suma de subconjuntos

Tenemos una mochila de un tamaño dado y objetos que ocupan cierto volumen. Queremos saber qué combinaciones de objetos llenan la mochila lo máximo posible sin pasarse.

## 1. Fuerza bruta

Probar todos los subconjuntos con `itertools.combinations` y quedarse con el que más llena sin pasarse. Cuesta $O(2^n)$.

In [1]:
from itertools import combinations

def mejor_subconjunto_fb(volumenes, capacidad):
    mejor = ()
    for r in range(len(volumenes) + 1):
        for combinacion in combinations(volumenes, r):
            if sum(mejor) < sum(combinacion) <= capacidad:
                mejor = combinacion
    return mejor

objetos = [5, 9, 13, 4, 7, 11]
print(mejor_subconjunto_fb(objetos, 30))

(5, 13, 11)


## 2. Programación dinámica

Optimizarlo: recorrer los objetos guardando, para cada suma alcanzable hasta la capacidad, un subconjunto que la consigue. Cuesta $O(n \cdot C)$ en lugar de $O(2^n)$.

In [2]:
import random
import timeit

def mejor_subconjunto(volumenes, capacidad):
    alcanzables = {0: ()}
    for v in volumenes:
        # list() evita modificar el dict mientras se recorre
        for suma, subconjunto in list(alcanzables.items()):
            if suma + v <= capacidad and suma + v not in alcanzables:
                alcanzables[suma + v] = subconjunto + (v,)
    return alcanzables[max(alcanzables)]

print(mejor_subconjunto(objetos, 30))
grandes = [random.randint(1, 100) for _ in range(20)]
assert sum(mejor_subconjunto(grandes, 500)) == sum(mejor_subconjunto_fb(grandes, 500))
for f in (mejor_subconjunto_fb, mejor_subconjunto):
    print(f"{f.__name__:22} {timeit.timeit(lambda: f(grandes, 500), number=1):.4f} s")

(9, 13, 7)
mejor_subconjunto_fb   0.1456 s
mejor_subconjunto      0.0004 s


## 3. Tabla booleana y reconstrucción

Construir la tabla $P[i][s]$, que dice si con los $i$ primeros objetos se puede sumar exactamente $s$: $P[i][s] = P[i-1][s] \lor P[i-1][s - v_i]$, con $P[0][0]$ cierto. Recorrerla hacia atrás para devolver un subconjunto que sume exactamente el objetivo (o `None` si no existe) y comprobarlo contra fuerza bruta.

In [3]:
from collections import Counter

def subconjunto_exacto(volumenes, objetivo):
    n = len(volumenes)
    P = [[False] * (objetivo + 1) for _ in range(n + 1)]
    P[0][0] = True
    for i, v in enumerate(volumenes, 1):
        for s in range(objetivo + 1):
            P[i][s] = P[i - 1][s] or (v <= s and P[i - 1][s - v])
    if not P[n][objetivo]:
        return None
    elegidos, s = [], objetivo
    for i in range(n, 0, -1):
        # si sin el objeto i no se llega a s, el objeto i está en la solución
        if not P[i - 1][s]:
            elegidos.append(volumenes[i - 1])
            s -= volumenes[i - 1]
    return elegidos[::-1]

def exacto_fuerza_bruta(volumenes, objetivo):
    return any(sum(c) == objetivo for r in range(len(volumenes) + 1) for c in combinations(volumenes, r))

print(subconjunto_exacto(objetos, 29), subconjunto_exacto(objetos, 3))

random.seed(13)
for _ in range(500):
    vols = [random.randint(1, 15) for _ in range(random.randint(0, 9))]
    objetivo = random.randint(0, 40)
    s = subconjunto_exacto(vols, objetivo)
    assert (s is not None) == exacto_fuerza_bruta(vols, objetivo)
    if s is not None:
        assert sum(s) == objetivo and not Counter(s) - Counter(vols)

[9, 13, 7] None


## 4. Contar las soluciones

Contar cuántos subconjuntos (por posición, aunque repitan valor) suman exactamente $S$. Basta cambiar el «o» por una suma: $N[s] \mathrel{+}= N[s - v]$, recorriendo $s$ de mayor a menor para usar cada objeto una vez. Comprobarlo contra fuerza bruta y calcular de cuántas formas se puede escribir 50 como suma de números distintos del 1 al 20.

In [4]:
def cuenta_subconjuntos(volumenes, objetivo):
    N = [1] + [0] * objetivo
    for v in volumenes:
        for s in range(objetivo, v - 1, -1):
            N[s] += N[s - v]
    return N[objetivo]

def cuenta_fuerza_bruta(volumenes, objetivo):
    return sum(sum(c) == objetivo for r in range(len(volumenes) + 1) for c in combinations(volumenes, r))

random.seed(14)
for _ in range(500):
    vols = [random.randint(1, 8) for _ in range(random.randint(0, 10))]
    objetivo = random.randint(0, 30)
    assert cuenta_subconjuntos(vols, objetivo) == cuenta_fuerza_bruta(vols, objetivo)

print(cuenta_subconjuntos(objetos, 29), cuenta_subconjuntos(range(1, 21), 50))

4 1969


## 5. Memoización frente a tabla

Resolver «¿se puede sumar exactamente $S$?» de arriba abajo, con una función recursiva `puede(i, s)` memoizada con `functools.cache`, y comparar cuántos estados calcula y cuánto tarda frente a la tabla completa de $(n + 1)(S + 1)$ casillas. La memoización solo visita los estados alcanzables desde $(n, S)$ y puede parar en cuanto encuentra una solución.

In [5]:
import time
from functools import cache

def exacto_memo(volumenes, objetivo):
    @cache
    def puede(i, s):
        if s == 0:
            return True
        if i == 0 or s < 0:
            return False
        return puede(i - 1, s - volumenes[i - 1]) or puede(i - 1, s)
    return puede(len(volumenes), objetivo), puede.cache_info().currsize

random.seed(15)
for _ in range(300):
    vols = [random.randint(1, 15) for _ in range(random.randint(0, 9))]
    objetivo = random.randint(0, 40)
    assert exacto_memo(vols, objetivo)[0] == exacto_fuerza_bruta(vols, objetivo)

vols = [random.randint(1, 200) for _ in range(40)]
for objetivo in (sum(vols) // 2, sum(vols) + 1):
    inicio = time.perf_counter()
    r_tabla = subconjunto_exacto(vols, objetivo) is not None
    t_tabla = time.perf_counter() - inicio
    inicio = time.perf_counter()
    r_memo, estados = exacto_memo(vols, objetivo)
    t_memo = time.perf_counter() - inicio
    assert r_tabla == r_memo
    print(f"S = {objetivo}: {r_memo}; tabla {(len(vols) + 1) * (objetivo + 1):>9,} casillas en {t_tabla * 1000:6.1f} ms,"
          f" memo {estados:>9,} estados en {t_memo * 1000:6.1f} ms")

S = 2058: True; tabla    84,419 casillas en    3.5 ms, memo     1,449 estados en    0.3 ms
S = 4117: False; tabla   168,838 casillas en    9.5 ms, memo    72,167 estados en   23.8 ms
